# 04 — Phase 2 Group Experiments
Evaluasi covariate groups yang lolos Phase 1 screening menggunakan 4 model dengan hyperparameter Optuna. Evaluasi: simple 80/20 temporal split.

In [ ]:
import sys, unittest.mock; sys.modules.setdefault('transformers.dependency_versions_check', unittest.mock.MagicMock())

import pandas as pd
import numpy as np
import joblib
import gc
import traceback
import warnings
from datetime import datetime

from darts import TimeSeries
from darts.models import RandomForestModel, XGBModel, LightGBMModel
from darts.dataprocessing.transformers import Scaler, Diff
from darts.utils.missing_values import fill_missing_values
from sklearn.ensemble import ExtraTreesRegressor

try:
    from darts.models import SKLearnModel
except ImportError:
    from darts.models import RegressionModel as SKLearnModel

warnings.filterwarnings("ignore")

LEVEL_VARS  = ["M2", "USDIDR", "Coal", "Copper", "Nickel", "Silver", "Tin", "STI", "Gold", "WTI", "GDP"]
RATE_VARS   = ["BI_Rate", "CPI", "NPL_Ratio", "US_Treasury_10Y"]
HORIZONS    = [1, 5, 20]   # sama dengan notebook 02
WINDOWS     = [20, 120]    # sama dengan notebook 02
TRAIN_RATIO = 0.8

import glob
joblib_files = sorted(glob.glob("saved_models/df_merged_*.joblib"), reverse=True)
df_merged = joblib.load(joblib_files[0])
print(f"Loaded: {joblib_files[0]}")
print(f"Data: {df_merged.shape} | {df_merged['date'].min().date()} to {df_merged['date'].max().date()}")

TUNING = joblib.load("saved_models/optuna_tuning_results.joblib")
print("\nOptuna best params:")
for m, r in TUNING.items():
    print(f"  {m}: MAPE={r['best_value']:.2f}  params={r['best_params']}")


In [ ]:

GROUP_COVARIATES = joblib.load("saved_models/phase2_groups.joblib")
GROUP_COVARIATES.pop("Sig_Regional", None)   # dikeluarkan — STI sudah masuk Sig_All

print("Phase 2 groups:")
for k, v in GROUP_COVARIATES.items():
    print(f"  {k}: {v}")

total_experiments = len(GROUP_COVARIATES) * 4 * len(WINDOWS) * len(HORIZONS)
print(f"\nTotal experiments: {len(GROUP_COVARIATES)} groups × 4 models × {len(WINDOWS)} windows × {len(HORIZONS)} horizons = {total_experiments}")


## Helper: to_series, build_model, evaluate_model

In [ ]:

def to_series(df, target_col, covariates=None):
    target = TimeSeries.from_dataframe(
        df, time_col="date", value_cols=target_col,
        fill_missing_dates=True, freq="B",
    )
    target = fill_missing_values(target)
    cov = None
    if covariates:
        cov = TimeSeries.from_dataframe(
            df, time_col="date", value_cols=covariates,
            fill_missing_dates=True, freq="B",
        )
        cov = fill_missing_values(cov)
    return target, cov


def build_model(model_name, window, horizon, has_covariates):
    best_params = TUNING[model_name]['best_params'].copy()
    common = {
        "lags": window,
        "lags_past_covariates": window if has_covariates else None,
        "output_chunk_length": horizon,
    }
    if model_name == "RandomForest":
        return RandomForestModel(**common, random_state=42, n_jobs=-1, **best_params)
    elif model_name == "ExtraTrees":
        return SKLearnModel(
            **common,
            model=ExtraTreesRegressor(random_state=42, n_jobs=-1, **best_params),
        )
    elif model_name == "XGBoost":
        return XGBModel(**common, random_state=42, n_jobs=-1, **best_params)
    elif model_name == "LightGBM":
        return LightGBMModel(**common, random_state=42, n_jobs=-1, verbose=-1, **best_params)
    else:
        raise ValueError(f"Unknown model: {model_name}")


def transform_target(target_ts, split_idx):
    train_ts  = target_ts[:split_idx]
    full_log  = target_ts.map(np.log)
    train_log = train_ts.map(np.log)
    diff = Diff(lags=1)
    train_log_diff = diff.fit_transform(train_log)
    full_log_diff  = diff.transform(full_log)
    scaler = Scaler()
    train_scaled = scaler.fit_transform(train_log_diff)
    full_scaled  = scaler.transform(full_log_diff)
    return train_scaled, full_scaled, scaler


def transform_covariates(cov_ts, split_idx):
    if cov_ts is None:
        return None, None
    train_cov = cov_ts[:split_idx]
    full_cov  = cov_ts
    cov_cols   = cov_ts.components.tolist()
    level_cols = [c for c in cov_cols if c in LEVEL_VARS]
    rate_cols  = [c for c in cov_cols if c in RATE_VARS]
    parts_train, parts_full = [], []
    if level_cols:
        d = Diff(lags=1)
        parts_train.append(d.fit_transform(train_cov[level_cols].map(np.log)))
        parts_full.append(d.transform(full_cov[level_cols].map(np.log)))
    if rate_cols:
        d = Diff(lags=1)
        parts_train.append(d.fit_transform(train_cov[rate_cols]))
        parts_full.append(d.transform(full_cov[rate_cols]))
    ct, cf = parts_train[0], parts_full[0]
    for pt, pf in zip(parts_train[1:], parts_full[1:]):
        ct = ct.stack(pt)
        cf = cf.stack(pf)
    cov_scaler = Scaler()
    cov_scaler.fit(ct)
    return cov_scaler.transform(cf), ct.end_time()


def inverse_and_metrics(forecast_list, full_ts, scaler, split_idx):
    full_log = full_ts.map(np.log)
    all_dates, all_prices = [], []
    for chunk_scaled in forecast_list:
        chunk_diff = scaler.inverse_transform(chunk_scaled)
        dates = chunk_diff.time_index
        vals  = chunk_diff.values().flatten()
        idx   = full_ts.get_index_at_point(dates[0])
        if idx == 0:
            continue
        anchor     = full_log[idx - 1].values()[0][0]
        log_prices = anchor + np.cumsum(vals)
        all_dates.extend(dates)
        all_prices.extend(np.exp(log_prices))

    pred_df   = pd.DataFrame({"date": pd.to_datetime(all_dates), "predicted": all_prices})
    actual_df = full_ts.to_dataframe().reset_index()
    actual_df.columns = ["date", "actual"]
    eval_df = pd.merge(actual_df, pred_df, on="date", how="inner")
    y_true  = eval_df["actual"].values
    y_pred  = eval_df["predicted"].values

    mape   = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    mae    = np.mean(np.abs(y_true - y_pred))
    rmse   = np.sqrt(np.mean((y_true - y_pred) ** 2))
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    r2     = 1 - (ss_res / ss_tot) if ss_tot > 0 else np.nan
    a_dir  = np.diff(y_true)
    p_dir  = y_pred[1:] - y_true[:-1]
    da     = np.mean((a_dir > 0) == (p_dir > 0)) * 100 if len(a_dir) > 0 else np.nan

    y_train_level = full_ts[:split_idx].values().flatten()
    naive_mae = np.mean(np.abs(np.diff(y_train_level))) if len(y_train_level) > 1 else np.nan
    mase = mae / naive_mae if (not np.isnan(naive_mae) and naive_mae > 0) else np.nan

    if len(y_true) > 2 and len(a_dir) > 0:
        returns_test = np.diff(y_true) / y_true[:-1]
        sigma        = np.std(returns_test)
        large_mask   = np.abs(returns_test) > sigma
        hit_large    = (np.mean((a_dir[large_mask] > 0) == (p_dir[large_mask] > 0)) * 100
                        if large_mask.sum() > 0 else np.nan)
    else:
        hit_large = np.nan

    return {
        "mape":      round(mape, 4),
        "mae":       round(mae, 4),
        "rmse":      round(rmse, 4),
        "r2":        round(r2, 4),
        "da":        round(da, 2),
        "mase":      round(mase, 4) if not np.isnan(mase) else np.nan,
        "hit_large": round(hit_large, 2) if not np.isnan(hit_large) else np.nan,
    }


def evaluate_model(model_name, cov_name, cov_vars, window, horizon):
    target_ts, cov_ts = to_series(df_merged, "IHSG", cov_vars if cov_vars else None)
    n         = len(target_ts)
    split_idx = int(n * TRAIN_RATIO)

    train_scaled, full_scaled, scaler = transform_target(target_ts, split_idx)
    full_cov_scaled, _ = transform_covariates(cov_ts, split_idx)

    model = build_model(model_name, window, horizon, has_covariates=bool(cov_vars))
    model.fit(train_scaled, past_covariates=full_cov_scaled)

    test_start    = target_ts[split_idx].start_time()
    forecast_list = model.historical_forecasts(
        series=full_scaled,
        past_covariates=full_cov_scaled,
        start=test_start,
        forecast_horizon=horizon,
        stride=horizon,
        retrain=False,
        last_points_only=False,
        verbose=False,
    )
    if isinstance(forecast_list, TimeSeries):
        forecast_list = [forecast_list]
    return inverse_and_metrics(forecast_list, target_ts, scaler, split_idx)


print("Helpers defined — HORIZONS:", HORIZONS, "| WINDOWS:", WINDOWS)


## Phase 2: Experiment Loop

In [ ]:

MODEL_NAMES = ["RandomForest", "ExtraTrees", "XGBoost", "LightGBM"]
results = []
failed  = []

total = len(MODEL_NAMES) * len(GROUP_COVARIATES) * len(WINDOWS) * len(HORIZONS)
done  = 0

for model_name in MODEL_NAMES:
    for cov_name, cov_vars in GROUP_COVARIATES.items():
        for window in WINDOWS:
            for horizon in HORIZONS:
                done += 1
                tag = f"{model_name} | {cov_name} | W{window}_H{horizon}"
                print(f"[{done}/{total}] {tag}", end=" ... ", flush=True)
                try:
                    m = evaluate_model(model_name, cov_name, cov_vars, window, horizon)
                    results.append({
                        "Model": model_name, "Covariates": cov_name,
                        "Window": window, "Horizon": horizon, **m
                    })
                    print(f"MAPE={m['mape']:.4f}%")
                except Exception as e:
                    failed.append({"tag": tag, "error": str(e)})
                    print(f"FAILED: {e}")
                    traceback.print_exc()
                finally:
                    gc.collect()

print(f"\nDone: {len(results)} OK, {len(failed)} failed")
if failed:
    print("Failed:", [f['tag'] for f in failed])


In [ ]:

df_p2 = pd.DataFrame(results)
df_p2.to_csv("phase2_group_results.csv", index=False)
print("Saved: phase2_group_results.csv")
print(df_p2.sort_values("mape").to_string(index=False))


In [ ]:

best = df_p2.sort_values("mape").iloc[0]
print("\nBest Phase 2 configuration:")
print(f"  Model      : {best['Model']}")
print(f"  Covariates : {best['Covariates']}")
print(f"  Window     : {best['Window']}")
print(f"  Horizon    : {best['Horizon']}")
print(f"  MAPE       : {best['mape']:.4f}%")
print(f"  RMSE       : {best['rmse']:.2f}")
print(f"  DA         : {best['da']:.1f}%")

best_config = best.to_dict()
joblib.dump(best_config, "saved_models/best_config_phase2.joblib")
print("\nSaved: saved_models/best_config_phase2.joblib")


## Baseline + Original Groups (Perbandingan)

Run 4 konfigurasi tambahan untuk melihat improvement setiap grup terhadap baseline:

| Grup | Isi |
|---|---|
| **Baseline** | Tanpa covariate (IHSG lags only) |
| **All_Macro** | Semua 7 variabel makro (termasuk US_Treasury_10Y) |
| **All_Commodity_Regional** | Semua 7 komoditas + STI (8 variabel) |
| **All_Covariates** | Semua 15 covariate |

Hasil digabung dengan Phase 2 (Sig_*) untuk tabel perbandingan lengkap.

In [ ]:
ALL_MACRO              = ["BI_Rate", "CPI", "M2", "NPL_Ratio", "USDIDR", "GDP", "US_Treasury_10Y"]
ALL_COMMODITY          = ["Coal", "Copper", "Nickel", "Silver", "Tin", "Gold", "WTI"]
ALL_COMMODITY_REGIONAL = ALL_COMMODITY + ["STI"]
ALL_COVARIATES         = ALL_MACRO + ALL_COMMODITY + ["STI"]

EXTRA_GROUPS = {
    "Baseline":               [],
    "All_Macro":              ALL_MACRO,
    "All_Commodity_Regional": ALL_COMMODITY_REGIONAL,
    "All_Covariates":         ALL_COVARIATES,
}

print("Extra groups to run:")
for k, v in EXTRA_GROUPS.items():
    print(f"  {k} ({len(v)} vars): {v}")

extra_results = []
extra_failed  = []
total_extra   = len(MODEL_NAMES) * len(EXTRA_GROUPS) * len(WINDOWS) * len(HORIZONS)
done = 0

for model_name in MODEL_NAMES:
    for cov_name, cov_vars in EXTRA_GROUPS.items():
        for window in WINDOWS:
            for horizon in HORIZONS:
                done += 1
                tag = f"{model_name} | {cov_name} | W{window}_H{horizon}"
                print(f"[{done}/{total_extra}] {tag}", end=" ... ", flush=True)
                try:
                    m = evaluate_model(model_name, cov_name, cov_vars, window, horizon)
                    extra_results.append({
                        "Model": model_name, "Covariates": cov_name,
                        "Window": window, "Horizon": horizon, **m
                    })
                    print(f"MAPE={m['mape']:.4f}%")
                except Exception as e:
                    extra_failed.append({"tag": tag, "error": str(e)})
                    print(f"FAILED: {e}")
                    traceback.print_exc()
                finally:
                    gc.collect()

print(f"\nDone: {len(extra_results)} OK, {len(extra_failed)} failed")
df_extra = pd.DataFrame(extra_results)
df_extra.to_csv("phase2_extra_results.csv", index=False)
print("Saved: phase2_extra_results.csv")


In [13]:
df_all = pd.concat([df_p2, df_extra], ignore_index=True)

# Baseline MAPE per model × window × horizon
baseline_mape = (
    df_all[df_all["Covariates"] == "Baseline"]
    .set_index(["Model", "Window", "Horizon"])["mape"]
    .rename("baseline_mape")
)

df_cmp = df_all.merge(
    baseline_mape.reset_index(), on=["Model", "Window", "Horizon"], how="left"
)
df_cmp["mape_impr_pct"] = (
    (df_cmp["baseline_mape"] - df_cmp["mape"]) / df_cmp["baseline_mape"] * 100
).round(3)
df_cmp["mape_impr_abs"] = (df_cmp["baseline_mape"] - df_cmp["mape"]).round(4)

GROUP_ORDER = [
    "Baseline",
    "Sig_Macro", "Sig_Commodity", "Sig_Regional", "Sig_All",
    "All_Macro", "All_Commodity_Regional", "All_Covariates",
]
df_cmp["grp_order"] = df_cmp["Covariates"].map(
    {g: i for i, g in enumerate(GROUP_ORDER)}
).fillna(99)
df_cmp = df_cmp.sort_values(["Horizon", "Model", "Window", "grp_order"])

# Ringkasan per grup × horizon (avg semua model × window)
print("=" * 75)
print("MAPE IMPROVEMENT VS BASELINE — rata-rata semua model × window, per horizon")
print("=" * 75)
for h in sorted(df_cmp["Horizon"].unique()):
    sub = df_cmp[(df_cmp["Covariates"] != "Baseline") & (df_cmp["Horizon"] == h)]
    summary = (
        sub.groupby("Covariates")
        .agg(avg_mape=("mape","mean"), avg_impr=("mape_impr_pct","mean"), best_mape=("mape","min"))
        .round(4)
    )
    summary["grp_order"] = summary.index.map({g: i for i, g in enumerate(GROUP_ORDER)}).fillna(99)
    summary = summary.sort_values("grp_order").drop(columns="grp_order")
    print(f"\nHorizon H{h}:")
    print(summary.to_string())

df_cmp.to_csv("phase2_full_comparison.csv", index=False)
print("\nSaved: phase2_full_comparison.csv")


MAPE IMPROVEMENT VS BASELINE — rata-rata semua model × window, per horizon

Horizon H1:
                        avg_mape  avg_impr  best_mape
Covariates                                           
Sig_Macro                 0.5065    0.1335     0.5019
Sig_Commodity             0.5064    0.1629     0.5040
Sig_All                   0.5058    0.2811     0.5040
All_Macro                 0.5086   -0.2841     0.5051
All_Commodity_Regional    0.5053    0.3648     0.5029
All_Covariates            0.5058    0.2671     0.5033

Horizon H5:
                        avg_mape  avg_impr  best_mape
Covariates                                           
Sig_Macro                 0.8516    0.2265     0.8383
Sig_Commodity             0.8435    1.1756     0.8336
Sig_All                   0.8435    1.1772     0.8263
All_Macro                 0.8594   -0.6836     0.8436
All_Commodity_Regional    0.8391    1.6934     0.8235
All_Covariates            0.8431    1.2262     0.8202

Horizon H20:
                     

In [14]:
print("=" * 65)
print("TOP 3 AKURASI PER SKENARIO (Window × Horizon)")
print("=" * 65)

df_all_clean = df_cmp.dropna(subset=["mape"]).copy()

for horizon in sorted(df_all_clean["Horizon"].unique()):
    for window in sorted(df_all_clean["Window"].unique()):
        mask = (df_all_clean["Window"] == window) & (df_all_clean["Horizon"] == horizon)
        sub  = df_all_clean[mask]

        # Baseline MAPE (rata-rata 4 model)
        bl_mape = sub[sub["Covariates"] == "Baseline"]["mape"].mean()

        # Top 3 non-baseline, diurutkan MAPE terendah
        top3 = (
            sub[sub["Covariates"] != "Baseline"]
            .sort_values("mape")
            .head(3)[["Model", "Covariates", "mape", "da", "mase", "mape_impr_pct"]]
            .reset_index(drop=True)
        )
        top3.index += 1

        print(f"\nW{window}_H{horizon}  |  Baseline MAPE avg = {bl_mape:.4f}%")
        print("-" * 65)
        for rank, row in top3.iterrows():
            impr = row["mape_impr_pct"]
            print(f"  #{rank}  {row['Model']:13s} | {row['Covariates']:22s} | "
                  f"MAPE={row['mape']:.4f}%  DA={row['da']:.1f}%  "
                  f"MASE={row['mase']:.4f}  Impr={impr:+.3f}%")

TOP 3 AKURASI PER SKENARIO (Window × Horizon)

W20_H1  |  Baseline MAPE avg = 0.5053%
-----------------------------------------------------------------
  #1  XGBoost       | Sig_Macro              | MAPE=0.5019%  DA=53.2%  MASE=0.9641  Impr=+0.417%
  #2  LightGBM      | Sig_Macro              | MAPE=0.5026%  DA=53.8%  MASE=0.9655  Impr=+0.396%
  #3  RandomForest  | Sig_All                | MAPE=0.5040%  DA=53.6%  MASE=0.9683  Impr=+0.631%

W120_H1  |  Baseline MAPE avg = 0.5091%
-----------------------------------------------------------------
  #1  XGBoost       | All_Commodity_Regional | MAPE=0.5029%  DA=54.4%  MASE=0.9662  Impr=+2.140%
  #2  LightGBM      | All_Covariates         | MAPE=0.5033%  DA=53.2%  MASE=0.9669  Impr=+1.294%
  #3  LightGBM      | All_Commodity_Regional | MAPE=0.5043%  DA=52.5%  MASE=0.9687  Impr=+1.098%

W20_H5  |  Baseline MAPE avg = 0.8531%
-----------------------------------------------------------------
  #1  XGBoost       | All_Covariates         | MAPE=0